<a href="https://colab.research.google.com/github/AnhQun18/Day21-Track3-Finetuning-Lab/blob/main/colab/Lab21_RUN_ALL.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lab 21 — Fine-tuning LLMs · RUN ALL (T4)

Chay tu tren xuong. Runtime > Change runtime type > **T4 GPU** truoc khi bat dau.

| O | Lam gi | Thoi gian |
|---|---|---|
| 1 | clone + install | ~1 phut |
| 2 | smoke: import + unit test | ~30 giay |
| 3 | **core pipeline NB1 -> NB5** | ~80 phut |
| 4 | gatekeeper + in ket qua | ~10 giay |


In [ ]:
# @title 1. Setup — clone + install (chạy ô này trước)
import os, subprocess, sys

REPO = "https://github.com/AnhQun18/Day21-Track3-Finetuning-Lab.git"
if not os.path.exists("Day21-Track3-Finetuning-Lab"):
    subprocess.run(["git", "clone", "-q", REPO], check=True)
os.chdir("/content/Day21-Track3-Finetuning-Lab")
subprocess.run(["git", "pull", "-q"], check=False)
sys.path.insert(0, "src")

# Install from requirements.txt, NOT a copied list. The copied list is how the
# torchao>=0.16 pin reached requirements.txt and this bootstrap on different days --
# and a bootstrap missing a pin does not fail here, it fails 10 minutes later inside
# get_peft_model(). One source of truth. torch is preinstalled on Colab and
# requirements.txt pins it compatibly, so that line is a no-op.
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"],
               check=True)

import torch
print("commit :", subprocess.run(["git","rev-parse","--short","HEAD"],
                                 capture_output=True, text=True).stdout.strip())
print("GPU    :", torch.cuda.get_device_name(0) if torch.cuda.is_available()
      else "NONE — Runtime > Change runtime type > T4 GPU")
if torch.cuda.is_available():
    print("VRAM   : %.1f GB" % (torch.cuda.get_device_properties(0).total_memory/1024**3))


commit : 8c657d9
GPU    : Tesla T4
VRAM   : 14.6 GB


In [ ]:
# @title 2. Smoke — imports, seed data, unit tests (no GPU needed)
!python scripts/verify.py --smoke



[  ok  ] labkit imports                                   
[  ok  ] tier resolves                                    T4 -> unsloth/Qwen3.5-4B
[  ok  ] all tiers respect the <32 effective-batch rule   
[  ok  ] data/train_seed.jsonl                            250 rows
[  ok  ] data/eval_target.jsonl                           50 rows
[  ok  ] data/eval_regression.jsonl                       15 rows
[  ok  ] unit tests                                       119 passed in 1.92s

7 passed · 0 warnings · 0 failures

Ready to submit.


In [4]:
# @title 3. Core pipeline — NB1 → NB5
# EVAL_LIMIT truncates both eval sets: "" = full run (submittable),
# 8 = ~fast smoke pass. STAGES lets you resume after a failure.
import os
COMPUTE_TIER = "T4"        # @param ["CPU","LAPTOP","T4","BIGGPU"]
EVAL_LIMIT   = "8"         # @param ["", "4", "8", "16", "25"]
STAGES       = "nb1 nb2 nb3 nb4 nb5"   # @param {type:"string"}

os.environ["COMPUTE_TIER"] = COMPUTE_TIER
if EVAL_LIMIT:
    os.environ["EVAL_LIMIT"] = EVAL_LIMIT
else:
    os.environ.pop("EVAL_LIMIT", None)

from labkit import device
print(device.banner(), "\n")

!python scripts/colab_run.py {STAGES}


Tesla T4 (cuda, sm_75, 14.6 GB) -> precision=fp16
  note: this GPU predates Ampere, so it has NO bfloat16. Using fp16 with gradient scaling instead. Tutorials that hardcode bf16=True fail here. 

tier=T4  mask=assistant-only  eval_limit=8

NB1 — data, chat template & loss mask
tier=T4  model=unsloth/Qwen3.5-4B  max_length=1024
250 mẫu huấn luyện
{
  "instruction": "Phân loại ticket chăm sóc khách hàng sau thành JSON với đúng 4 khóa: intent, urgency, product, sentiment. Chỉ trả về JSON, không giải thích.\n\nintent thuộc: doi_tra | van_chuyen | hoan_tien | san_pham_loi | hoi_thong_tin\nurgency thuộc: cao | trung_binh | thap\nsentiment thuộc: tieu_cuc | trung_tinh | tich_cuc\nproduct: tên sản phẩm xuất hiện trong ticket.",
  "input": "Alo sh
config.json: 100% 2.76k/2.76k [00:00<00:00, 9.13MB/s]
tokenizer_config.json: 100% 15.7k/15.7k [00:00<00:00, 29.9MB/s]

tokenizer.json: downloading bytes:   0% 0.00/20.0M [00:00<?, ?B/s]
tokenizer.json: downloading bytes: 100% 6.23M/6.23M [00:00<00:00,

In [5]:
# @title 4. Gatekeeper + results
!python scripts/verify.py
print("\n================ results/ ================")
!ls -la results/
!echo && echo "---- runs.csv ----" && cat results/runs.csv 2>/dev/null
!echo && echo "---- verdict.json ----" && cat results/verdict.json 2>/dev/null



[  ok  ] labkit imports                                   
[  ok  ] tier resolves                                    T4 -> unsloth/Qwen3.5-4B
[  ok  ] all tiers respect the <32 effective-batch rule   
[  ok  ] data/train_seed.jsonl                            250 rows
[  ok  ] data/eval_target.jsonl                           50 rows
[  ok  ] data/eval_regression.jsonl                       15 rows
[ FAIL ] unit tests                                       1 failed, 118 passed in 2.59s
[  ok  ] results/template_check.json                      
[  ok  ] results/mask_proof.json                          
[  ok  ] results/token_stats.json                         
[  ok  ] results/baselines_frozen.json                    
[  ok  ] results/runs.csv                                 
[  ok  ] results/verdict.json                             
[  ok  ] results/autopsy.json                             
[  ok  ] submission/REPORT.md                             
[ FAIL ] REPORT.md filled in           

In [9]:
# Full evaluation recovery; preserve all original prompts and evaluation data
from pathlib import Path
import os, subprocess, sys, shutil
p=Path('tests/test_repo_structure.py')
s=p.read_text()
old='"Lab21_RUN_ALL cell 1": "".join(run_all["cells"][1]["source"]),'
new='"Lab21_RUN_ALL setup cell": next("".join(c["source"]) for c in run_all["cells"] if c.get("cell_type") == "code"),'
assert old in s
p.write_text(s.replace(old,new))
for f,var,names in [('notebooks/02_baselines.py','baseline_predictions.json',['preds_a','preds_b','rpreds_b']),('notebooks/05_evaluate_and_verdict.py','ft_predictions.json',['preds_ft','rpreds_ft'])]:
 p=Path(f); s=p.read_text()
 addition='\nreport.write_json({'+','.join(repr(n)+': '+n for n in names)+'}, '+repr(var)+', results_dir=ROOT / "results")\n'
 if var not in s: p.write_text(s+addition)
os.environ.pop('EVAL_LIMIT',None)
os.environ['COMPUTE_TIER']='T4'
# Keep .env aligned with the full evaluation request.
p=Path('.env')
if p.exists(): p.write_text('\n'.join(l for l in p.read_text().splitlines() if not l.strip().startswith('EVAL_LIMIT='))+'\n')
r=subprocess.run([sys.executable,'-m','pytest','tests/','--tb=short','-rN'])
assert r.returncode==0
subprocess.run([sys.executable,'-u','scripts/colab_run.py','nb2','nb5'],check=True)


AssertionError: 

In [ ]:
import subprocess, sys, os
r=subprocess.run([sys.executable,'scripts/build_colab.py'],capture_output=True,text=True)
print(r.stdout, r.stderr)
r=subprocess.run([sys.executable,'-m','pytest','tests/','--tb=short','-rN'],capture_output=True,text=True)
print(r.stdout, r.stderr)
assert r.returncode==0
os.environ.pop('EVAL_LIMIT',None)
subprocess.run([sys.executable,'-u','scripts/colab_run.py','nb2','nb5'],check=True)


wrote 6 notebooks to colab/:
   Lab21_01_data_and_mask.ipynb
   Lab21_02_baselines.ipynb
   Lab21_03_train_correct.ipynb
   Lab21_04_misconfig_autopsy.ipynb
   Lab21_05_evaluate_and_verdict.ipynb
   Lab21_06_merge_and_serve.ipynb
 
........................................................................ [ 60%]
...............................................                          [100%]
119 passed in 1.88s
 


In [7]:
# Diagnostic: inspect requirements and the failing test
from pathlib import Path
import subprocess, sys
for f in ['Makefile','submission/REPORT.md','scripts/run_all.py','scripts/verify.py']:
 p=Path(f)
 if p.exists():
  print('\n===== '+f+' =====\n'+p.read_text())
r=subprocess.run([sys.executable,'-m','pytest','-q','--tb=short'],capture_output=True,text=True)
print(r.stdout[-16000:])
print(r.stderr[-3000:])


===== Makefile =====
# Lab 21 — Fine-tuning LLMs (Track 3)
.DEFAULT_GOAL := help
PY ?= python3
VENV ?= .venv
BIN := $(VENV)/bin

.PHONY: help setup setup-cpu smoke nb1 nb2 nb3 nb4 nb5 nb6 pipeline pipeline-full test verify colab data clean

help:  ## Show this help
	@grep -E '^[a-zA-Z_-]+:.*?## .*$$' $(MAKEFILE_LIST) | awk 'BEGIN{FS=":.*?## "}{printf "  \033%-16s\033 %s\n", $$1, $$2}'

setup:  ## Full GPU install (torch first — see requirements.txt header)
	$(PY) -m venv $(VENV) && $(BIN)/pip install -U pip && $(BIN)/pip install -r requirements.txt
	@echo "Now: cp .env.example .env && make smoke"

setup-cpu:  ## CPU-only install — enough for NB1 + the whole test suite, no GPU
	$(PY) -m venv $(VENV) && $(BIN)/pip install -U pip && $(BIN)/pip install -r requirements-cpu.txt

smoke:  ## Imports + data + unit tests. No GPU, no model download.
	$(BIN)/python scripts/verify.py --smoke

data:  ## Regenerate the seed corpus (deterministic)
	$(BIN)/python scripts/make_seed_data.py

nb1:  ## NB